# Semana 10 — Gabarito Revisado dos Exercícios de SQL Avançado para Data Warehouse

Este gabarito foi revisado para ficar coerente com os slides finais da Semana 10.

## Ajustes importantes

- `tempo_sk` é uma chave substituta, não uma data.
- Para filtros de período, use `dim_tempo.data`, `dim_tempo.ano`, `dim_tempo.mes` ou `dim_tempo.ano_mes`.
- Para exemplos de índice no DuckDB, use tabelas físicas com sufixo `_base`.
- A Aula 3 foi tratada como teórica: particionamento e materialized views aparecem como interpretação conceitual, não como prática obrigatória em DuckDB.

## 0. Preparação do ambiente

In [3]:
import duckdb
import pandas as pd

# Ajuste o caminho dos arquivos se necessário.
# Se os CSVs estiverem na mesma pasta do notebook, este código já deve funcionar.

dim_cliente = pd.read_csv('../Dataset_schema/dim_cliente.csv')
dim_produto = pd.read_csv('../Dataset_schema/dim_produto.csv')
dim_tempo = pd.read_csv('../Dataset_schema/dim_tempo.csv')
fato_vendas = pd.read_csv('../Dataset_schema/fato_vendas.csv')

banco_dados = duckdb.connect()

# Registra os DataFrames como tabelas consultáveis pelo DuckDB
banco_dados.register('dim_cliente', dim_cliente)
banco_dados.register('dim_produto', dim_produto)
banco_dados.register('dim_tempo', dim_tempo)
banco_dados.register('fato_vendas', fato_vendas)

# Para a parte de índices, precisamos de tabelas físicas no DuckDB.
# Por isso criamos versões com o sufixo _base.
banco_dados.sql("DROP TABLE IF EXISTS fato_vendas_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_cliente_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_produto_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_tempo_base")

banco_dados.sql("CREATE TABLE fato_vendas_base AS SELECT * FROM fato_vendas")
banco_dados.sql("CREATE TABLE dim_cliente_base AS SELECT * FROM dim_cliente")
banco_dados.sql("CREATE TABLE dim_produto_base AS SELECT * FROM dim_produto")
banco_dados.sql("CREATE TABLE dim_tempo_base AS SELECT * FROM dim_tempo")

---

# Parte 1 — Subconsultas

## Exercício 1 — Vendas acima da média geral

In [ ]:
query = """
SELECT
    venda_sk,
    order_id,
    cliente_sk,
    valor_total
FROM fato_vendas
WHERE valor_total > (
    SELECT AVG(valor_total)
    FROM fato_vendas
)
ORDER BY valor_total DESC;
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 2 — Vendas com frete acima da média geral

In [ ]:
query = """
SELECT
    venda_sk,
    order_id,
    valor_frete,
    valor_total
FROM fato_vendas
WHERE valor_frete > (
    SELECT AVG(valor_frete)
    FROM fato_vendas
)
ORDER BY valor_frete DESC;
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 3 — Categorias com quantidade de vendas acima da média

In [4]:
query = """
SELECT
    categoria,
    qtd_vendas
FROM (
    SELECT
        p.product_category_name AS categoria,
        COUNT(*) AS qtd_vendas
    FROM fato_vendas f
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    WHERE p.product_category_name IS NOT NULL
    GROUP BY p.product_category_name
) categorias
WHERE qtd_vendas > (
    SELECT AVG(qtd_vendas)
    FROM (
        SELECT
            COUNT(*) AS qtd_vendas
        FROM fato_vendas f
        INNER JOIN dim_produto p
            ON f.produto_sk = p.produto_sk
        WHERE p.product_category_name IS NOT NULL
        GROUP BY p.product_category_name
    ) media_categorias
)
ORDER BY qtd_vendas DESC;
"""

# Para executar:
banco_dados.sql(query).df()

,categoria,qtd_vendas
0,cama_mesa_banho,10953
1,beleza_saude,9465
2,esporte_lazer,8431
3,moveis_decoracao,8160
4,informatica_acessorios,7644
5,utilidades_domesticas,6795
6,relogios_presentes,5859
7,telefonia,4430
8,ferramentas_jardim,4268
9,automotivo,4140


## Exercício 4 — Vendas dos 3 produtos mais vendidos

In [ ]:
query = """
SELECT
    venda_sk,
    order_id,
    produto_sk,
    valor_total
FROM fato_vendas
WHERE produto_sk IN (
    SELECT
        produto_sk
    FROM fato_vendas
    GROUP BY produto_sk
    ORDER BY COUNT(*) DESC
    LIMIT 3
)
ORDER BY produto_sk, valor_total DESC;
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 5 — Estados com receita acima da média dos estados

In [6]:
query = """
SELECT
    receita_estado.estado,
    ROUND(receita_estado.receita_total, 2) AS receita_total
FROM (
    SELECT
        c.customer_state AS estado,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    GROUP BY c.customer_state
) receita_estado
WHERE receita_estado.receita_total > (
    SELECT AVG(receita_total)
    FROM (
        SELECT
            c.customer_state AS estado,
            SUM(f.valor_total) AS receita_total
        FROM fato_vendas f
        INNER JOIN dim_cliente c
            ON f.cliente_sk = c.cliente_sk
        GROUP BY c.customer_state
    ) media_estado
)
ORDER BY receita_total DESC;
"""

# Para executar:
banco_dados.sql(query).df()

,estado,receita_total
0,SP,5769703.15
1,RJ,2055401.57
2,MG,1818891.67
3,RS,861472.79
4,PR,781708.80
5,SC,595127.78
6,BA,591137.81


---

# Parte 2 — CTEs com WITH

## Exercício 6 — Vendas por estado usando CTE

In [7]:
query = """
WITH vendas_por_estado AS (
    SELECT
        c.customer_state AS estado,
        COUNT(DISTINCT f.order_id) AS qtd_pedidos,
        COUNT(*) AS qtd_itens_vendidos,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    GROUP BY c.customer_state
)
SELECT
    estado,
    qtd_pedidos,
    qtd_itens_vendidos,
    ROUND(receita_total, 2) AS receita_total
FROM vendas_por_estado
ORDER BY receita_total DESC;
"""

# Para executar:
banco_dados.sql(query).df()

,estado,qtd_pedidos,qtd_itens_vendidos,receita_total
0,SP,40501,46448,5769703.15
1,RJ,12350,14143,2055401.57
2,MG,11354,12916,1818891.67
3,RS,5345,6134,861472.79
4,PR,4923,5649,781708.80
5,SC,3546,4097,595127.78
6,BA,3256,3683,591137.81
7,DF,2080,2355,346123.35
8,GO,1957,2277,334212.35
9,ES,1995,2225,317657.93


## Exercício 7 — Categorias com maior faturamento usando CTE

In [8]:
query = """
WITH resumo_categorias AS (
    SELECT
        p.product_category_name AS categoria,
        COUNT(*) AS quantidade_vendas,
        SUM(f.valor_total) AS receita_total,
        AVG(f.valor_total) AS ticket_medio
    FROM fato_vendas f
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    WHERE p.product_category_name IS NOT NULL
    GROUP BY p.product_category_name
)
SELECT
    categoria,
    quantidade_vendas,
    ROUND(receita_total, 2) AS receita_total,
    ROUND(ticket_medio, 2) AS ticket_medio
FROM resumo_categorias
WHERE receita_total > 100000
ORDER BY receita_total DESC;
"""

# Para executar:
banco_dados.sql(query).df()

,categoria,quantidade_vendas,receita_total,ticket_medio
0,beleza_saude,9465,1412089.53,149.19
1,relogios_presentes,5859,1264333.12,215.79
2,cama_mesa_banho,10953,1225209.26,111.86
3,esporte_lazer,8431,1118256.91,132.64
4,informatica_acessorios,7644,1032723.77,135.10
5,moveis_decoracao,8160,880329.92,107.88
6,utilidades_domesticas,6795,758392.25,111.61
7,cool_stuff,3718,691680.89,186.04
8,automotivo,4140,669454.75,161.70
9,ferramentas_jardim,4268,567145.68,132.88


## Exercício 8 — Receita por estado e categoria usando CTE

In [9]:
query = """
WITH receita_por_estado_categoria AS (
    SELECT
        c.customer_state AS estado,
        p.product_category_name AS categoria,
        COUNT(*) AS quantidade_vendas,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    WHERE p.product_category_name IS NOT NULL
    GROUP BY
        c.customer_state,
        p.product_category_name
)
SELECT
    estado,
    categoria,
    quantidade_vendas,
    ROUND(receita_total, 2) AS receita_total
FROM receita_por_estado_categoria
WHERE receita_total > 50000
ORDER BY
    estado,
    receita_total DESC;
"""

# Para executar:
banco_dados.sql(query).df()

,estado,categoria,quantidade_vendas,receita_total
0,BA,beleza_saude,340,58620.07
1,BA,relogios_presentes,225,50254.70
2,MG,beleza_saude,1064,175007.52
3,MG,cama_mesa_banho,1322,156483.10
4,MG,relogios_presentes,626,132311.65
...,...,...,...,...
60,SP,construcao_ferramentas_construcao,422,66397.74
61,SP,consoles_games,472,65789.05
62,SP,fashion_bolsas_e_acessorios,841,65487.98
63,SP,pcs,59,64371.71


## Exercício 9 — Meses com receita acima da média mensal

In [11]:
query = """
WITH receita_mensal AS (
    SELECT
        t.ano_mes,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    INNER JOIN dim_tempo t
        ON f.tempo_sk = t.tempo_sk
    GROUP BY t.ano_mes
),
media_mensal AS (
    SELECT
        AVG(receita_total) AS media_receita_mensal
    FROM receita_mensal
)
SELECT
    r.ano_mes,
    ROUND(r.receita_total, 2) AS receita_total
FROM receita_mensal r
CROSS JOIN media_mensal m
WHERE r.receita_total > m.media_receita_mensal
ORDER BY r.ano_mes;
"""

# Para executar:
banco_dados.sql(query).df()

,ano_mes,receita_total
0,2017-09,701077.49
1,2017-10,751117.01
2,2017-11,1153364.20
3,2017-12,843078.29
4,2018-01,1077887.46
5,2018-02,966168.41
6,2018-03,1120598.24
7,2018-04,1132878.93
8,2018-05,1128774.52
9,2018-06,1011978.29


## Exercício 10 — Pipeline de análise por categoria

In [13]:
query = """
WITH metricas_categoria AS (
    SELECT
        p.product_category_name AS categoria,
        COUNT(*) AS qtd_vendas,
        SUM(f.valor_total) AS receita_total,
        AVG(f.valor_total) AS ticket_medio
    FROM fato_vendas f
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    WHERE p.product_category_name IS NOT NULL
    GROUP BY p.product_category_name
)
SELECT
    categoria,
    qtd_vendas,
    ROUND(receita_total, 2) AS receita_total,
    ROUND(ticket_medio, 2) AS ticket_medio
FROM metricas_categoria
WHERE receita_total > 50000
  AND ticket_medio > 100
ORDER BY receita_total DESC;
"""

# Para executar:
banco_dados.sql(query).df()

,categoria,qtd_vendas,receita_total,ticket_medio
0,beleza_saude,9465,1412089.53,149.19
1,relogios_presentes,5859,1264333.12,215.79
2,cama_mesa_banho,10953,1225209.26,111.86
3,esporte_lazer,8431,1118256.91,132.64
4,informatica_acessorios,7644,1032723.77,135.10
5,moveis_decoracao,8160,880329.92,107.88
6,utilidades_domesticas,6795,758392.25,111.61
7,cool_stuff,3718,691680.89,186.04
8,automotivo,4140,669454.75,161.70
9,ferramentas_jardim,4268,567145.68,132.88


---

# Parte 3 — Window Functions

## Exercício 11 — Ranking das vendas por categoria

In [15]:
query = """
SELECT
    p.product_category_name AS categoria,
    f.order_id,
    f.valor_total,
    ROW_NUMBER() OVER (
        PARTITION BY p.product_category_name
        ORDER BY f.valor_total DESC
    ) AS ranking_venda_categoria
FROM fato_vendas f
INNER JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
WHERE p.product_category_name IS NOT NULL
ORDER BY
    categoria,
    ranking_venda_categoria;
"""

# Para executar:
banco_dados.sql(query).df()

,categoria,order_id,valor_total,ranking_venda_categoria
0,agro_industria_e_comercio,d7a2c0c1ff66b314f3bf166fb4157fd4,3184.55,1
1,agro_industria_e_comercio,9f5abaa01d419a15e448f089b1cf4b94,2467.33,2
2,agro_industria_e_comercio,c205abf27ff07fddd3fa9a67037138e5,2315.83,3
3,agro_industria_e_comercio,193d998b23a996345ca322516b2d5af3,1978.18,4
4,agro_industria_e_comercio,23aee6ac2f5a0a3056aa62c615974c0c,1518.55,5
...,...,...,...,...
108655,utilidades_domesticas,baa821bd1dbd5c6d31c9b1c68f760b8b,11.85,6791
108656,utilidades_domesticas,8fbcb92faf1aa60361f61ed7ae721a7e,6.08,6792
108657,utilidades_domesticas,8fbcb92faf1aa60361f61ed7ae721a7e,6.08,6793
108658,utilidades_domesticas,8fbcb92faf1aa60361f61ed7ae721a7e,6.08,6794


## Exercício 12 — Pedido mais caro de cada estado

In [16]:
query = """
WITH ranking_pedidos_estado AS (
    SELECT
        c.customer_state AS estado,
        f.order_id,
        f.valor_total,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_state
            ORDER BY f.valor_total DESC
        ) AS posicao
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
)
SELECT
    estado,
    order_id,
    ROUND(valor_total, 2) AS valor_total
FROM ranking_pedidos_estado
WHERE posicao = 1
ORDER BY valor_total DESC;
"""

# Para executar:
banco_dados.sql(query).df()

,estado,order_id,valor_total
0,MS,0812eb902a67711a1cb742b3cdaa65ae,6929.31
1,ES,fefacc66af859508bf1a7934eab1e97f,6922.21
2,SP,f5136e38d1a14a4dbd87dff67da82701,6726.66
3,RJ,a96610ab360d42a2e5335a3998b4718a,4950.34
4,PB,8dbc85d1447242f3b127dda390d56e19,4681.78
5,DF,80dfedb6d17bf23539beeef3c768f4d7,4194.76
6,MG,68101694e5c5dc7330c91e1bbc36214f,4175.26
7,PA,9a3966c23190dbdbaabed08e8429c006,4042.74
8,PE,d3f66901a6743e15f9311547cc623b91,3792.59
9,RS,f398a143c0fe171d965db2096cf064cf,3297.40


## Exercício 13 — Ranking de estados por receita

In [18]:
query = """
WITH receita_por_estado AS (
    SELECT
        c.customer_state AS estado,
        COUNT(DISTINCT f.order_id) AS qtd_pedidos,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    GROUP BY c.customer_state
)
SELECT
    estado,
    qtd_pedidos,
    ROUND(receita_total, 2) AS receita_total,
    RANK() OVER (
        ORDER BY receita_total DESC
    ) AS ranking_estado
FROM receita_por_estado
ORDER BY ranking_estado;
"""

# Para executar:
banco_dados.sql(query).df()

,estado,qtd_pedidos,receita_total,ranking_estado
0,SP,40501,5769703.15,1
1,RJ,12350,2055401.57,2
2,MG,11354,1818891.67,3
3,RS,5345,861472.79,4
4,PR,4923,781708.80,5
5,SC,3546,595127.78,6
6,BA,3256,591137.81,7
7,DF,2080,346123.35,8
8,GO,1957,334212.35,9
9,ES,1995,317657.93,10


## Exercício 14 — Comparando RANK e DENSE_RANK com empate

In [ ]:
query = """
WITH exemplo_receita_categoria AS (
    SELECT 'beleza_saude' AS categoria, 100000 AS receita_total
    UNION ALL
    SELECT 'cama_mesa_banho', 90000
    UNION ALL
    SELECT 'esporte_lazer', 90000
    UNION ALL
    SELECT 'informatica_acessorios', 70000
    UNION ALL
    SELECT 'moveis_decoracao', 60000
)
SELECT
    categoria,
    receita_total,
    RANK() OVER (
        ORDER BY receita_total DESC
    ) AS rank_categoria,
    DENSE_RANK() OVER (
        ORDER BY receita_total DESC
    ) AS dense_rank_categoria
FROM exemplo_receita_categoria
ORDER BY receita_total DESC;
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 15 — Percentual da venda dentro da categoria

In [4]:
query = """
SELECT
    p.product_category_name AS categoria,
    f.order_id,
    f.valor_total,
    SUM(f.valor_total) OVER (
        PARTITION BY p.product_category_name
    ) AS total_categoria,
    ROUND(
        100.0 * f.valor_total / SUM(f.valor_total) OVER (
            PARTITION BY p.product_category_name
        ),
        2
    ) AS percentual_na_categoria
FROM fato_vendas f
INNER JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
WHERE p.product_category_name IS NOT NULL
ORDER BY
    categoria,
    percentual_na_categoria DESC;
"""

# Para executar:
banco_dados.sql(query).df()

,categoria,order_id,valor_total,total_categoria,percentual_na_categoria
0,agro_industria_e_comercio,d7a2c0c1ff66b314f3bf166fb4157fd4,3184.55,76203.30,4.18
1,agro_industria_e_comercio,9f5abaa01d419a15e448f089b1cf4b94,2467.33,76203.30,3.24
2,agro_industria_e_comercio,c205abf27ff07fddd3fa9a67037138e5,2315.83,76203.30,3.04
3,agro_industria_e_comercio,193d998b23a996345ca322516b2d5af3,1978.18,76203.30,2.60
4,agro_industria_e_comercio,23aee6ac2f5a0a3056aa62c615974c0c,1518.55,76203.30,1.99
...,...,...,...,...,...
108655,utilidades_domesticas,54c9556bfdeffbc7cc3558ab954114f6,29.32,758392.25,0.00
108656,utilidades_domesticas,50a980068f11953231ad1d03fb1769b4,35.79,758392.25,0.00
108657,utilidades_domesticas,2c8bf07fcf5c57d898ff1797efa33379,29.63,758392.25,0.00
108658,utilidades_domesticas,2c476f0148a99ce31bffe7659e5aaf4d,37.79,758392.25,0.00


## Exercício 16 — Receita acumulada mensal por estado

In [5]:
query = """
WITH receita_mensal_estado AS (
    SELECT
        c.customer_state AS estado,
        t.ano,
        t.mes,
        t.ano_mes,
        SUM(f.valor_total) AS receita_mes
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    INNER JOIN dim_tempo t
        ON f.tempo_sk = t.tempo_sk
    GROUP BY
        c.customer_state,
        t.ano,
        t.mes,
        t.ano_mes
)
SELECT
    estado,
    ano,
    mes,
    ano_mes,
    ROUND(receita_mes, 2) AS receita_mes,
    ROUND(
        SUM(receita_mes) OVER (
            PARTITION BY estado
            ORDER BY ano, mes
        ),
        2
    ) AS receita_acumulada_estado
FROM receita_mensal_estado
ORDER BY
    estado,
    ano,
    mes;
"""

# Para executar:
banco_dados.sql(query).df()

,estado,ano,mes,ano_mes,receita_mes,receita_acumulada_estado
0,AC,2017,1,2017-01,723.14,723.14
1,AC,2017,2,2017-02,597.40,1320.54
2,AC,2017,3,2017-03,530.18,1850.72
3,AC,2017,4,2017-04,1351.51,3202.23
4,AC,2017,5,2017-05,2371.73,5573.96
...,...,...,...,...,...,...
551,TO,2018,4,2018-04,5370.51,45245.41
552,TO,2018,5,2018-05,3314.33,48559.74
553,TO,2018,6,2018-06,4987.03,53546.77
554,TO,2018,7,2018-07,3795.09,57341.86


---

# Parte 4 — Índices e Otimização

## Exercício 17 — Índice para filtro por estado

In [6]:
query = """
-- Índice sugerido
CREATE INDEX idx_dim_cliente_estado
ON dim_cliente_base (customer_state);

-- Consulta que pode se beneficiar do índice
SELECT
    cliente_sk,
    customer_id,
    customer_state
FROM dim_cliente_base
WHERE customer_state = 'SP';
"""

# Para executar:
banco_dados.sql(query).df()

,cliente_sk,customer_id,customer_state
0,1,06b8999e2fba1a1fbc88172c00ba8bc7,SP
1,2,18955e83d337fd6b2def6b18a428ac77,SP
2,3,4e7b3e00288586ebd08712fdd0374a03,SP
3,4,b2b6027bc5c5109e529d4dc6358b12c3,SP
4,5,4f2d8ab171c80ec8364f7c12e35b23ad,SP
...,...,...,...
41741,99433,f255d679c7c86c24ef4861320d5b7675,SP
41742,99435,f5a0b560f9e9427792a88bec97710212,SP
41743,99437,17ddf5dd5d51696bb3d7c6291687be6f,SP
41744,99438,e7b71a9017aa05c9a7fd292d714858e8,SP


## Exercício 18 — Índice para JOIN e filtro por data

In [ ]:
query = """
-- Índice na chave de junção da tabela fato
CREATE INDEX idx_fato_vendas_tempo_sk
ON fato_vendas_base (tempo_sk);

-- Índice na chave de junção da dimensão tempo
CREATE INDEX idx_dim_tempo_tempo_sk
ON dim_tempo_base (tempo_sk);

-- Índice na coluna usada no filtro de data
CREATE INDEX idx_dim_tempo_data
ON dim_tempo_base (data);

-- Consulta que pode se beneficiar dos índices
SELECT
    f.venda_sk,
    f.order_id,
    f.valor_total,
    t.data
FROM fato_vendas_base f
INNER JOIN dim_tempo_base t
    ON f.tempo_sk = t.tempo_sk
WHERE t.data >= '2018-01-01';
"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 19 — Índice composto para filtro combinado

In [ ]:
query = """
CREATE INDEX idx_fato_vendas_cliente_produto
ON fato_vendas_base (cliente_sk, produto_sk);

SELECT
    venda_sk,
    order_id,
    cliente_sk,
    produto_sk,
    valor_total
FROM fato_vendas_base
WHERE cliente_sk = 5000
  AND produto_sk = 15420;
"""

# Para executar:
# banco_dados.sql(query).df()

---

# Parte 5 — Aula 3 teórica: particionamento e materialized views

## Exercício 20 — Análise conceitual de performance em DW

### Resposta esperada

1. A consulta pode ficar lenta porque precisa ler muitos registros da tabela fato, fazer `JOIN` com várias dimensões e ainda agrupar os dados com `GROUP BY`. Em bases grandes, isso aumenta o custo de leitura, junção e agregação.

2. Particionar a tabela fato por uma coluna de data real, como `data_venda`, ajuda porque o banco pode ler apenas as partes do período consultado. Por exemplo, uma consulta de 2018 não precisaria percorrer todo o histórico.

3. Uma Materialized View com receita por mês, estado e categoria guarda o resultado agregado fisicamente. Assim, o dashboard consulta um resumo pronto, em vez de recalcular todos os `JOINs` e agregações a cada abertura.

4. `tempo_sk` não deve ser tratado como data neste dataset porque é uma chave substituta sequencial. Ela serve para relacionar `fato_vendas` com `dim_tempo`. Para análise temporal, devemos usar `data`, `ano`, `mes` ou `ano_mes`.

---

# Observações para correção

Considere corretas as respostas que:

- usam os nomes reais das tabelas e colunas;
- fazem os `JOINs` pelas chaves corretas;
- usam `product_category_name` ou `product_category_name_english` de forma consistente, quando o arquivo possuir ambas;
- entendem que `tempo_sk` é chave substituta;
- usam CTEs, subconsultas e Window Functions quando o enunciado solicitar;
- nos exercícios de índice, identificam corretamente as colunas do `WHERE` e do `JOIN`.